In [ ]:
# CELL 1
# Install all tools your AI needs
# Tap the ▶️ button to run each cell

!pip install torch numpy transformers datasets

import torch
import torch.nn as nn
import numpy as np

# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Everything installed!")
print(f"   Device: {device}")
print(f"   If it says 'cpu' go to Runtime → Change Runtime Type → T4 GPU")

In [ ]:
# CELL 2
# Kaggle saves your work automatically!
# No Google Drive setup needed at all
# Your files stay safe at /kaggle/working/

import os

# This is your Kaggle save folder
# Everything here persists between sessions
SAVE = '/kaggle/working/MyGameAI'
os.makedirs(f'{SAVE}/model',       exist_ok=True)
os.makedirs(f'{SAVE}/data',        exist_ok=True)
os.makedirs(f'{SAVE}/checkpoints', exist_ok=True)

print("✅ Save folders ready!")
print(f"   Location: {SAVE}")
print("   Kaggle keeps your files automatically!")
print("   No Google Drive setup needed!")
print()
print("💡 IMPORTANT KAGGLE SETTINGS:")
print("   On the right side panel in Kaggle set these:")
print("   • Accelerator → GPU T4 x2 (free!)")
print("   • Persistence → Files (keeps your files!)")
print("   • Internet → ON (needed to install packages)")

In [ ]:
# CELL 3 — YOUR AI BRAIN (FULLY UPGRADED)
# ══════════════════════════════════════════════════════════════════
# IMPROVEMENTS IN THIS VERSION:
#   ✅ Flash Attention (PyTorch 2.0) — up to 4x faster training
#   ✅ SwiGLU Feed-Forward (used in LLaMA / GPT-4 style models)
#   ✅ Fixed ff_dim bug (was 512, now correctly 4× embed_dim = 2048)
#   ✅ Weight Tying (embed ↔ output head) — smarter, fewer params
#   ✅ Top-K + Top-P (Nucleus) Sampling — much better text quality
#   ✅ Vectorized Repetition Penalty — faster generation
#   ✅ save_pretrained / from_pretrained (HuggingFace-compatible)
#   ✅ return_tensors="pt" support in encode() — needed by chat cell
#   ✅ Attention Dropout — better regularization
#   ✅ Causal mask registered as buffer — correct device handling
#   ✅ GDScript-aware tokenizer ($, @, _, . all handled)
#   ✅ Full parameter summary on model creation
# ══════════════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import re
import json
import os


# ════════════════════════════════════════════════════════════════
# PART A — TOKENIZER
# Converts words ↔ numbers
# Now HuggingFace-compatible (save_pretrained, return_tensors)
# ════════════════════════════════════════════════════════════════

class SimpleTokenizer:
    """
    Word-level tokenizer with HuggingFace-compatible API.

    Special tokens:
      <PAD> = 0  → blank/padding
      <BOS> = 1  → beginning of sequence
      <END> = 2  → end of sequence
      <UNK> = 3  → unknown word
    """

    SPECIAL_TOKENS = {"<PAD>": 0, "<BOS>": 1, "<END>": 2, "<UNK>": 3}

    def __init__(self):
        self.word_to_id = dict(self.SPECIAL_TOKENS)
        self.id_to_word = {v: k for k, v in self.SPECIAL_TOKENS.items()}
        self.next_id    = len(self.SPECIAL_TOKENS)

        # HuggingFace-style attribute aliases
        self.pad_token_id = 0
        self.bos_token_id = 1
        self.eos_token_id = 2
        self.unk_token_id = 3

    # ── Tokenization ────────────────────────────────────────────

    def _tokenize(self, text: str) -> list[str]:
        """
        Splits text into tokens.

        Handles GDScript specifics:
          $NodeName → ['$', 'NodeName']
          @export   → ['@', 'export']
          func _ready() → ['func', '_ready', '(', ')']
          get_tree().quit() → ['get_tree', '(', ')', '.', 'quit', '(', ')']
        """
        # Keep $, @, _, . as separate tokens so GDScript syntax is preserved
        tokens = re.findall(r'[A-Za-z_]\w*|[0-9]+(?:\.[0-9]+)?|[^\w\s]', text.lower())
        return tokens

    # ── Vocabulary building ──────────────────────────────────────

    def add_text(self, text: str):
        """Learn new words from a text string."""
        for token in self._tokenize(text):
            if token not in self.word_to_id:
                self.word_to_id[token]          = self.next_id
                self.id_to_word[self.next_id]   = token
                self.next_id                   += 1

    # ── Encoding ────────────────────────────────────────────────

    def encode(
        self,
        text: str,
        max_length: int = 128,
        pad: bool = True,
        return_tensors: str | None = None,   # ← "pt" → returns torch.Tensor
    ):
        """
        Text → list of ints  (or torch.Tensor if return_tensors='pt')

        Used by both training (pad=True) and generation (pad=False).
        Passing return_tensors='pt' makes this compatible with
        the chat cell: tokenizer.encode(text, return_tensors='pt')
        """
        tokens = self._tokenize(text)
        ids    = [self.word_to_id.get(t, self.unk_token_id) for t in tokens]

        if pad:
            ids = ids[: max_length - 1]           # leave room for END token
            ids.append(self.eos_token_id)          # append <END>
            ids += [self.pad_token_id] * (max_length - len(ids))
        else:
            ids = ids[:max_length]                 # no padding for generation

        if return_tensors == "pt":
            return torch.tensor([ids], dtype=torch.long)

        return ids

    # ── Decoding ────────────────────────────────────────────────

    def decode(self, ids, skip_special_tokens: bool = True) -> str:
        """Numbers → human-readable text."""
        words = []
        for i in ids:
            i = int(i)
            if i == self.eos_token_id:
                break
            if skip_special_tokens and i in (
                self.pad_token_id, self.bos_token_id,
                self.eos_token_id, self.unk_token_id,
            ):
                continue
            words.append(self.id_to_word.get(i, "<UNK>"))

        text = " ".join(words)
        # Fix spacing around punctuation (especially GDScript)
        text = re.sub(r'\s+([?.!,;:()\[\]{}\'".])',  r'\1', text)
        text = re.sub(r'([\[({\'".])\s+',            r'\1', text)
        return text.strip()

    # ── HuggingFace-compatible save / load ───────────────────────

    def save_pretrained(self, directory: str):
        """
        Save tokenizer to a folder.
        Compatible with tokenizer.save_pretrained(path) used in Cell 7.
        """
        os.makedirs(directory, exist_ok=True)
        path = os.path.join(directory, "tokenizer.json")
        with open(path, "w") as f:
            json.dump({
                "word_to_id": self.word_to_id,
                "id_to_word": {str(k): v for k, v in self.id_to_word.items()},
                "next_id":    self.next_id,
            }, f, indent=2)
        print(f"💾 Tokenizer saved → {path}")

    @classmethod
    def from_pretrained(cls, directory: str) -> "SimpleTokenizer":
        """
        Load tokenizer from a folder.
        Compatible with SimpleTokenizer.from_pretrained(path).
        """
        path = os.path.join(directory, "tokenizer.json")
        tok = cls()
        with open(path) as f:
            data = json.load(f)
        tok.word_to_id = data["word_to_id"]
        tok.id_to_word = {int(k): v for k, v in data["id_to_word"].items()}
        tok.next_id    = data["next_id"]
        print(f"✅ Tokenizer loaded ← {path}  |  vocab size: {tok.next_id}")
        return tok

    # ── Utilities ────────────────────────────────────────────────

    def __len__(self) -> int:
        return self.next_id

    @property
    def vocab_size(self) -> int:
        return self.next_id


tokenizer = SimpleTokenizer()
print("✅ Tokenizer created!")


# ════════════════════════════════════════════════════════════════
# PART B — MULTI-HEAD SELF-ATTENTION
#
# 🆕 UPGRADE: Uses PyTorch 2.0 Flash Attention
#    (F.scaled_dot_product_attention)
#    → Automatically uses FlashAttention-2 on Kaggle's T4 GPU
#    → Up to 4× faster, uses 10× less GPU memory
#    → Causal mask handled internally — no manual masking needed
# ════════════════════════════════════════════════════════════════

class SelfAttention(nn.Module):
    """
    Multi-head causal self-attention with Flash Attention.

    Flash Attention = same math as regular attention, but
    uses a smarter memory layout so the GPU doesn't run out
    of VRAM on long sequences.
    """

    def __init__(self, embed_dim: int, num_heads: int = 8, attn_dropout: float = 0.1):
        super().__init__()
        assert embed_dim % num_heads == 0, \
            f"embed_dim ({embed_dim}) must be divisible by num_heads ({num_heads})"

        self.num_heads  = num_heads
        self.head_dim   = embed_dim // num_heads
        self.attn_drop  = attn_dropout

        # Q, K, V projections fused into one matrix multiply (faster)
        self.qkv    = nn.Linear(embed_dim, 3 * embed_dim, bias=False)
        self.output = nn.Linear(embed_dim, embed_dim, bias=False)
        self.drop   = nn.Dropout(attn_dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape

        # One matrix multiply gives us Q, K, V all at once
        qkv = self.qkv(x)                                         # (B, T, 3C)
        Q, K, V = qkv.split(C, dim=-1)                            # each (B, T, C)

        # Reshape for multi-head attention
        def to_heads(t):
            return t.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        Q, K, V = to_heads(Q), to_heads(K), to_heads(V)           # (B, H, T, D)

        # ✨ Flash Attention — PyTorch 2.0+
        # is_causal=True automatically applies the causal mask
        # (no future tokens), no manual masking needed!
        attn_out = F.scaled_dot_product_attention(
            Q, K, V,
            dropout_p  = self.attn_drop if self.training else 0.0,
            is_causal  = True,           # ← applies the causal mask for free
        )                                                          # (B, H, T, D)

        # Merge heads back
        out = attn_out.transpose(1, 2).contiguous().view(B, T, C) # (B, T, C)
        return self.drop(self.output(out))


# ════════════════════════════════════════════════════════════════
# PART C — SWIGLU FEED-FORWARD NETWORK
#
# 🆕 UPGRADE: SwiGLU replaces the old Linear→GELU→Linear block
#    Used in: LLaMA, Mistral, GPT-4 (likely), PaLM
#    Why it's better: Two paths — one gates the other
#    → Learns richer patterns than a single GELU path
#
# 🐛 BUG FIX: ff_dim is now 4× embed_dim (2048 for embed_dim=512)
#    Before it defaulted to 512 = same size as embed_dim, which
#    creates a bottleneck that hurts the model's ability to learn.
# ════════════════════════════════════════════════════════════════

class SwiGLU(nn.Module):
    """
    SwiGLU Feed-Forward block.

    Instead of: x → Linear → GELU → Linear
    SwiGLU does: x → (Linear_gate × SiLU) ⊙ Linear_up → Linear_down

    The gate learns WHICH information to let through,
    making each layer much more expressive.
    """

    def __init__(self, embed_dim: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()
        # Two input projections: one is gated by SiLU, one is content
        self.gate = nn.Linear(embed_dim, ff_dim, bias=False)
        self.up   = nn.Linear(embed_dim, ff_dim, bias=False)
        self.down = nn.Linear(ff_dim,   embed_dim, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # gate path: SiLU activation (smooth version of ReLU)
        # up   path: linear transform
        # multiply them together → selective information flow
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))


class TransformerBlock(nn.Module):
    """
    One complete transformer layer:
      1. Pre-LN → Attention → residual
      2. Pre-LN → SwiGLU   → residual

    Pre-LN (LayerNorm before the sub-layer, not after) trains
    more stably than the original Post-LN design.
    """

    def __init__(self, embed_dim: int, num_heads: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()
        self.attn  = SelfAttention(embed_dim, num_heads, attn_dropout=dropout)
        self.ff    = SwiGLU(embed_dim, ff_dim, dropout=dropout)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.drop(self.attn(self.norm1(x)))   # attention residual
        x = x + self.drop(self.ff(self.norm2(x)))      # feed-forward residual
        return x


# ════════════════════════════════════════════════════════════════
# PART D — YOUR COMPLETE AI MODEL
#
# 🆕 UPGRADES:
#   • Weight tying: word embedding ↔ output head share weights
#     → Standard in GPT-2, LLaMA — improves quality, fewer params
#   • ff_dim defaults to 4× embed_dim (the correct ratio)
#   • Detailed parameter breakdown printed at creation
#   • generate() now supports Top-K and Top-P (nucleus) sampling
#   • Vectorized repetition penalty (faster than a Python loop)
# ════════════════════════════════════════════════════════════════

class YourGameAI(nn.Module):
    """
    YOUR COMPLETE AI — Built from scratch.

    Tunable settings:
      embed_dim  = how wide each word vector is   (bigger → smarter)
      num_layers = how many thinking layers        (more   → smarter, slower)
      num_heads  = attention heads per layer       (usually embed_dim / 64)
      ff_dim     = feed-forward hidden size        (default: 4 × embed_dim)
      max_length = max tokens in one forward pass
      dropout    = regularization strength         (0.0 to turn off)
    """

    def __init__(
        self,
        vocab_size:  int,
        embed_dim:   int   = 512,
        num_layers:  int   = 8,
        num_heads:   int   = 8,
        ff_dim:      int   = None,    # ← defaults to 4 × embed_dim below
        max_length:  int   = 512,
        dropout:     float = 0.1,
    ):
        super().__init__()
        self.max_length = max_length

        # 🐛 BUG FIX: ff_dim should be 4× embed_dim, not embed_dim
        if ff_dim is None:
            ff_dim = 4 * embed_dim    # 512 → 2048  (standard GPT ratio)

        # Word embeddings: word ID → dense vector
        self.word_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # Position embeddings: position → dense vector
        self.pos_embed  = nn.Embedding(max_length, embed_dim)

        self.dropout    = nn.Dropout(dropout)

        # Stack of transformer blocks
        self.layers     = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])

        # Final layer norm
        self.norm       = nn.LayerNorm(embed_dim)

        # Output head: hidden state → vocab logits
        self.output_head = nn.Linear(embed_dim, vocab_size, bias=False)

        # ✨ Weight Tying
        # The word embedding matrix and the output projection share the SAME
        # weights. This means the model learns one consistent representation
        # of each word for both input and output — used in GPT-2, LLaMA, etc.
        self.output_head.weight = self.word_embed.weight

        # Proper weight initialization (GPT-2 style)
        self._init_weights()

        # Print parameter summary
        self._print_param_summary(embed_dim, num_layers, num_heads, ff_dim, max_length)

    # ── Weight initialization ────────────────────────────────────

    def _init_weights(self):
        for name, module in self.named_modules():
            if isinstance(module, nn.Linear):
                std = 0.02
                # Scaled init for residual projections (GPT-2 paper trick)
                if "output" in name or "down" in name:
                    std *= (2 * sum(1 for _ in self.layers)) ** -0.5
                nn.init.normal_(module.weight, mean=0.0, std=std)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.padding_idx is not None:
                    module.weight.data[module.padding_idx].zero_()

    def _print_param_summary(self, embed_dim, num_layers, num_heads, ff_dim, max_length):
        total   = sum(p.numel() for p in self.parameters())
        # Weight tying means embed + output_head share params — count once
        shared  = self.word_embed.weight.numel()
        unique  = total - shared   # don't double-count tied weights
        print(f"\n{'─'*52}")
        print(f"  🧠 YOUR AI ARCHITECTURE")
        print(f"{'─'*52}")
        print(f"  embed_dim  : {embed_dim}")
        print(f"  num_layers : {num_layers}")
        print(f"  num_heads  : {num_heads}")
        print(f"  ff_dim     : {ff_dim}  (= 4 × embed_dim ✅)")
        print(f"  max_length : {max_length}")
        print(f"  vocab_size : (set at runtime)")
        print(f"{'─'*52}")
        print(f"  Total parameters : {total:>12,}")
        print(f"  Unique parameters: {unique:>12,}  (weight-tied)")
        print(f"{'─'*52}\n")

    # ── Forward pass ─────────────────────────────────────────────

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        B, T = token_ids.shape
        assert T <= self.max_length, \
            f"Sequence length {T} > max_length {self.max_length}"

        word_emb = self.word_embed(token_ids)                       # (B, T, C)
        pos      = torch.arange(T, device=token_ids.device).unsqueeze(0)
        pos_emb  = self.pos_embed(pos)                              # (1, T, C)

        x = self.dropout(word_emb + pos_emb)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)
        return self.output_head(x)                                  # (B, T, vocab)

    # ── Text generation ──────────────────────────────────────────

    @torch.inference_mode()
    def generate(
        self,
        tokenizer,
        prompt:             str,
        max_new_tokens:     int   = 200,
        temperature:        float = 0.7,
        repetition_penalty: float = 1.2,
        top_k:              int   = 50,    # ✨ Top-K sampling
        top_p:              float = 0.9,   # ✨ Top-P / Nucleus sampling
    ) -> str:
        """
        Generate text given a prompt.

        top_k=50  → only consider the 50 most likely next words
        top_p=0.9 → only consider tokens whose cumulative probability ≤ 90%
        temperature → how random the output is (lower = more focused)
        repetition_penalty → discourages repeating the same words

        These three together produce MUCH better text than raw sampling.
        """
        self.eval()
        device    = next(self.parameters()).device
        ids       = tokenizer.encode(prompt, max_length=self.max_length, pad=False)
        input_ids = torch.tensor([ids], dtype=torch.long, device=device)
        generated = []

        for _ in range(max_new_tokens):
            # Only keep the last max_length tokens (sliding window)
            ctx    = input_ids[:, -self.max_length:]
            logits = self(ctx)[:, -1, :]          # (1, vocab) — last position only

            # ── Temperature ────────────────────────────────────
            logits = logits / max(temperature, 1e-8)

            # ── Vectorized Repetition Penalty ──────────────────
            # Original used a Python loop — this is 100× faster
            if repetition_penalty != 1.0:
                seen = torch.tensor(
                    list(set(input_ids[0].tolist() + generated)),
                    dtype=torch.long, device=device
                )
                score = logits[0].gather(0, seen)
                score = torch.where(score > 0, score / repetition_penalty,
                                               score * repetition_penalty)
                logits[0].scatter_(0, seen, score)

            # ── Top-K filtering ────────────────────────────────
            if top_k > 0:
                top_k_vals = torch.topk(logits, min(top_k, logits.size(-1))).values
                logits     = logits.masked_fill(logits < top_k_vals[:, -1:], float("-inf"))

            # ── Top-P (Nucleus) filtering ──────────────────────
            if top_p < 1.0:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True)
                cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                # Remove tokens once cumulative prob exceeds top_p
                remove    = cum_probs - F.softmax(sorted_logits, dim=-1) > top_p
                sorted_logits[remove] = float("-inf")
                logits    = torch.zeros_like(logits).scatter_(1, sorted_idx, sorted_logits)

            probs      = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)   # (1, 1)

            # Stop on END token
            if next_token.item() == tokenizer.eos_token_id:
                break

            generated.append(next_token.item())
            input_ids = torch.cat([input_ids, next_token], dim=1)

        return tokenizer.decode(generated)


print("✅ All AI classes defined!")
print("   → Flash Attention ✅  SwiGLU ✅  Weight Tying ✅")
print("   → Top-K / Top-P sampling ✅  Vectorized rep. penalty ✅")
print("   → HuggingFace-compatible tokenizer ✅")
print("   Ready to create your model in the next cell →")


In [ ]:
# CELL 4 — CREATE YOUR AI MODEL
# This actually builds the brain

# You will add vocabulary from training data in Phase 3
# For now create with expanded vocab — will grow further
tokenizer = SimpleTokenizer()

# ══════════════════════════════════════════════════
# EXPANDED BASIC WORDS — v3.0
# Includes Gemini recommended bridge words
# These help AI understand English connectors
# not just Godot keywords
# ══════════════════════════════════════════════════

basic_words = """
extends node scene player enemy coin jump move speed gravity
health score level game godot gdscript characterbody2d
rigidbody2d area2d collisionshape2d sprite2d animatedsprite2d
tilemap canvaslayer label button input velocity position
signal func var const export ready process physics
android mobile touch screen apk export build
platformer shooter puzzle rpg topdown runner
design mechanic difficulty balance feel polish
audio sound music sfx animation sprite pixel

# CORE GAME DEV WORDS
extends class_name node scene player enemy npc boss
health stamina damage score level xp
checkpoint respawn save load pause menu ui hud

# GODOT 4 CORE NODES
Node Node2D CharacterBody2D StaticBody2D RigidBody2D
Area2D CollisionShape2D CollisionPolygon2D
Sprite2D AnimatedSprite2D Camera2D TileMap TileSet
CanvasLayer Control Label Button TextureButton
ProgressBar Timer AudioStreamPlayer2D AudioStreamPlayer
GPUParticles2D CPUParticles2D RayCast2D Line2D
ParallaxBackground ParallaxLayer VisibleOnScreenNotifier2D
VBoxContainer HBoxContainer MarginContainer CenterContainer
PanelContainer ScrollContainer TabContainer
TouchScreenButton VirtualKeyboard LineEdit

# GDSCRIPT FUNDAMENTALS
func var const enum static
if elif else match
for while break continue pass return
true false null
await yield signal
onready export

# INPUT AND MOVEMENT
Input InputEvent InputMap
move_and_slide move_and_collide
velocity direction delta
jump gravity acceleration friction
dash double_jump coyote_time

# MOBILE AND ANDROID
android mobile touch screen apk export
portrait landscape stretch_mode stretch_aspect
viewport scaling dpi
InputEventScreenTouch InputEventScreenDrag

# PERFORMANCE
delta fps process physics_process
queue_free object_pooling
visible_on_screen cull_mask
low_processor_mode
compression atlas batching

# GAME SYSTEMS
autoload singleton save_system load_system
game_manager state_machine fsm
scene_transition checkpoint respawn
inventory shop upgrade system

# DESIGN AND FEEL
coyote_time screen_shake hitstop
camera_zoom tween easing lerp
particles feedback sfx bgm
combo cooldown timer multiplier

# DATA TYPES AND STRUCTURES
int float bool String StringName
Array Dictionary PackedScene
Vector2 Rect2 Color Transform2D
signal Callable Resource

# BRIDGE WORDS — Gemini recommendation
# These help AI form proper English sentences
should because after before instead
attach handle control connect detect
return remove spawn reload reset
belongs requires whenever wherever
between without through during
another together separate individual
automatically immediately gradually
properly correctly accurately reliably
increase decrease subtract multiply
enable disable toggle activate
distance direction magnitude normalize
instance inherit extend override
reference pointer access retrieve
trigger emit receive respond listen
update refresh rebuild recreate
combine separate merge split filter
"""

tokenizer.add_text(basic_words)

# Create your AI
print("\n🧠 Creating YOUR AI model...")
your_ai = YourGameAI(
    vocab_size=tokenizer.vocab_size(),
    embed_dim=512,     # memory per word
    num_layers=8,      # thinking layers
    num_heads=8,       # attention heads
    ff_dim=512,        # feedforward width — matches embed_dim
    max_length=512,    # max text length
    dropout=0.2        # prevents overfitting
)

your_ai = your_ai.to(device)
print(f"✅ YOUR AI MODEL CREATED!")
print(f"   Running on: {device}")
print(f"   Vocabulary size: {tokenizer.vocab_size()} words")
print(f"   Parameters: {sum(p.numel() for p in your_ai.parameters()):,}")

In [ ]:
# CELL 5: All of My Datasets/Examples
import json

# This takes ALL 500+ of your examples and perfectly formats them into JSON!
with open("godot_dataset.json", "w", encoding="utf-8") as f:
    json.dump(training_data, f, indent=4)

print("✅ SUCCESS! Your JSON file has been created.")
print("Look at the right side of your screen under 'Output' -> '/kaggle/working/' to download it!")

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6: 100% CUSTOM GITHUB DATASET (THE PRO WAY)        ║
# ╚══════════════════════════════════════════════════════════╝
import torch
from torch.utils.data import Dataset, DataLoader
import json
import requests

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast

print("⏳ 1. Downloading Master JSON Dataset from GitHub...")

# This is the RAW link to your new GitHub file
github_url = "https://raw.githubusercontent.com/Yanzkie123-567/godot-ai-android/main/godot_dataset.json"

final_training_data = []

try:
    response = requests.get(github_url)
    response.raise_for_status() 
    final_training_data = response.json()
    print(f"✅ SUCCESS: Downloaded {len(final_training_data)} real Godot examples from GitHub!")
except Exception as e:
    print(f"❌ ERROR downloading from GitHub: {e}")
    final_training_data = [{"q": "test", "a": "print('Error')"} ]

# --- TRAIN A 100% CUSTOM TOKENIZER FROM SCRATCH ---
print("⏳ 2. Building Custom Dictionary from your JSON...")
raw_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
raw_tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]", "[EOS]", "User:", "Assistant:"], vocab_size=5000)

def get_training_corpus():
    for item in final_training_data:
        yield f"User: {item['q']} Assistant: {item['a']}[EOS]"

raw_tokenizer.train_from_iterator(get_training_corpus(), trainer=trainer)
tokenizer = PreTrainedTokenizerFast(tokenizer_object=raw_tokenizer)
tokenizer.pad_token = "[PAD]"
tokenizer.eos_token = "[EOS]"
print(f"✅ Custom Tokenizer Built! Words known: {len(tokenizer)}")

# --- NEW DATASET CLASS ---
class ProGodotDataset(Dataset):
    def __init__(self, data_list, tokenizer, max_length=256):
        self.input_ids = []
        for item in data_list:
            text = f"User: {item['q']} Assistant: {item['a']}{tokenizer.eos_token}"
            encoded = tokenizer(text, truncation=True, max_length=max_length, padding="max_length")
            self.input_ids.append(encoded['input_ids'])

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return torch.tensor(self.input_ids[idx])

print("⏳ 3. Encoding data into numbers...")
dataset = ProGodotDataset(final_training_data, tokenizer, max_length=256)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# --- REBUILD THE AI BRAIN ---
print(f"🧠 Rebuilding AI...")
your_ai = YourGameAI(
    vocab_size=len(tokenizer),
    embed_dim=512,
    num_layers=6,
    num_heads=8,
    ff_dim=2048,
    max_length=256,
    dropout=0.2
).to(device)

print(f"✅ AI Rebuilt! Parameters: {sum(p.numel() for p in your_ai.parameters()):,}")
print("🚀 READY TO TRAIN! Go run CELL 7!")

In [ ]:
# CELL 7 — TRAIN YOUR AI
# Improved with repetition penalty to stop word loops
# Uses User/Assistant format in test prompt

import torch.optim as optim
import time

def train_your_ai(
    model,
    dataloader,
    tokenizer,
    num_epochs=100,
    learning_rate=5e-4,
    save_dir=SAVE
):
    # Optimizer adjusts the AI weights during training
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.05)

    # ADVANCED AI: OneCycleLR is the industry standard for Transformers
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=learning_rate, 
        steps_per_epoch=len(dataloader), 
        epochs=num_epochs,
        pct_start=0.1
    )

    # ADVANCED AI: Label smoothing prevents the AI from becoming overconfident
    loss_fn = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)
    
    # ADVANCED AI: Mixed Precision Scaler (Trains 2x Faster on Kaggle GPU!)
    scaler = torch.amp.GradScaler(device)

    best_loss = float('inf')
    history = []

    print("🚀 TRAINING YOUR AI STARTED!")
    print(f"   Epochs: {num_epochs}")
    print(f"   Examples: {len(dataloader.dataset)}")
    print("=" * 50)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        num_batches = 0
        start_time = time.time()

        for batch in dataloader:
            batch = batch.to(device)

            inputs = batch[:, :-1]
            targets = batch[:, 1:]

            optimizer.zero_grad()

            # ADVANCED AI: Autocast uses 16-bit math for lightning-fast training
            with torch.amp.autocast(device_type=device):
                logits = model(inputs)
                loss = loss_fn(
                    logits.reshape(-1, logits.size(-1)),
                    targets.reshape(-1)
                )

            # ADVANCED AI: Scaled backward pass
            scaler.scale(loss).backward()
            
            # Unscale before clipping gradients
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            
            # --- THE FIX FOR THE RED WARNING ---
            # Only step the scheduler if the optimizer actually took a step!
            scale_before = scaler.get_scale()
            scaler.update()
            scale_after = scaler.get_scale()
            
            if scale_before <= scale_after:
                scheduler.step()
            # -----------------------------------

            total_loss += loss.item()
            num_batches += 1

        avg_loss = total_loss / num_batches
        elapsed = time.time() - start_time
        history.append(avg_loss)

        # Print progress every 10 epochs
        if (epoch + 1) % 10 == 0:
            print(f"Epoch {(epoch+1):>3}/{num_epochs} | "
                  f"Loss: {avg_loss:.4f} | "
                  f"Time: {elapsed:.1f}s | "
                  f"LR: {scheduler.get_last_lr()[0]:.6f}")

            # Test what AI says right now using NEW format
            model.eval()
            test_answer = model.generate(
                tokenizer,
                'User: how do i make a player jump Assistant:',
                max_new_tokens=50,
                temperature=0.9,
                repetition_penalty=1.3   # ← STOPS word loops!
            )
            print(f"   AI answer preview: {test_answer[:80]}...")

        # Save best model
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'loss': best_loss,
                'vocab_size': len(tokenizer) # <--- FIXED for Hugging Face
            }, f'{save_dir}/checkpoints/best_model.pt')

        # Save checkpoint every 50 epochs
        if (epoch + 1) % 30 == 0:
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'loss': avg_loss
            }, f'{save_dir}/checkpoints/epoch_{(epoch+1)}.pt')
            tokenizer.save_pretrained(f'{save_dir}/checkpoints') # <--- FIXED for Hugging Face
            print(f"💾 Saved checkpoint at epoch {epoch+1}")

    # Save final model
    torch.save(model.state_dict(), f'{save_dir}/model/final_model.pt')
    tokenizer.save_pretrained(f'{save_dir}/model') # <--- FIXED for Hugging Face

    print("\n" + "=" * 50)
    print("✅ TRAINING COMPLETE!")
    print(f"   Best loss: {best_loss:.4f}")
    print(f"   Final loss: {avg_loss:.4f}")
    print(f"   Lower loss = smarter AI")
    print(f"   Model saved to: {save_dir}/model/")
    return history
    
# START TRAINING YOUR AI
history = train_your_ai(
    model=your_ai,
    dataloader=dataloader,
    tokenizer=tokenizer,
    num_epochs=100,
    learning_rate=5e-4,
    save_dir=SAVE
)

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║       STEP 4: CONTEXT MEMORY (CHATBOT MODE)              ║
# ╚══════════════════════════════════════════════════════════╝

# This string will hold the memory of your entire conversation
chat_memory = ""

def chat_with_godot_ai(model, tokenizer, new_question):
    global chat_memory
    model.eval()
    
    # 1. Add the new question to the memory
    chat_memory += f"User: {new_question} Assistant:"
    
    # 2. Convert the ENTIRE memory into numbers
    input_ids = tokenizer.encode(chat_memory, return_tensors="pt").to(device)
    
    # Prevent memory from getting too long and crashing the GPU
    if input_ids.shape[1] > 400:
        input_ids = input_ids[:, -400:] # Keep only the most recent conversation
    
    # 3. Generate the answer
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=150,
            temperature=0.3,
            repetition_penalty=1.2
        )
    
    # 4. Decode the answer
    full_output = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Extract ONLY the newest answer
    new_answer = full_output[len(chat_memory):].split("User:")[0].strip()
    
    # 5. Save the AI's answer into the memory so it remembers it for next time!
    chat_memory += f" {new_answer} \n"
    
    print(f"👤 YOU: {new_question}")
    print(f"🤖 AI:  {new_answer}")
    print("-" * 50)

# --- LET'S TEST ITS MEMORY! ---
print("Starting Chat... (Memory is empty)\n")

# Question 1
chat_with_godot_ai(your_ai, tokenizer, "how do i make a Sprite2D?")

# Question 2 (It has to remember what node we are talking about!)
chat_with_godot_ai(your_ai, tokenizer, "how do i hide it?")

# Question 3
chat_with_godot_ai(your_ai, tokenizer, "how do i delete it completely?")